# GLM-5 / Kimi Distillation: GRPO + Eval + Publish

**Runs on Colab Pro H100** (80GB VRAM). Everything happens here — no Tinker API calls.

## Full Pipeline
1. Download SFT LoRA weights from Tinker (just a file download)
2. GRPO reinforcement learning on top of SFT (Unsloth + TRL)
3. Benchmark ALL models with lm-evaluation-harness
4. Merge best model + push to HuggingFace + GGUF export

## Benchmarks (lm-eval, industry standard)
| Benchmark | Shots | Method | Task |
|-----------|-------|--------|------|
| GSM8K | 8-shot CoT | Generative | `gsm8k_cot` |
| MATH | 4-shot | Generative (\\boxed{}) | `minerva_math` |
| ARC-Challenge | 25-shot | Log-likelihood | `arc_challenge` |
| GPQA Diamond | 0-shot | Log-likelihood | `gpqa_diamond` |
| MMLU-Pro | 5-shot | Log-likelihood | `mmlu_pro` |

In [ ]:
# Cell 0: Install everything + mount Drive
!pip install unsloth vllm lm-eval tinker
!pip install --upgrade pillow

from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json
DRIVE = '/content/drive/MyDrive/distillreasoning'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/eval_results', exist_ok=True)
print(f'Drive dir: {DRIVE}')

In [ ]:
# Cell 1: Download SFT LoRA weights from Tinker
# This is just a file download — no GPU usage, no credits burned
import tinker, requests, tarfile

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
service = tinker.ServiceClient()
rest = service.create_rest_client()

SFT_CHECKPOINTS = {
    'glm5': 'tinker://0fbca836-2aae-5500-b28d-93c2a46a328b:train:0/sampler_weights/qwen35-4b-glm5-final',
    'kimi': 'tinker://f5795e66-71e4-5cf4-9ebe-1cc14c27aa6e:train:0/sampler_weights/qwen35-4b-kimi-final',
    'combined': 'tinker://41e7bd9e-e49f-5f13-a5ff-f4339faab448:train:0/sampler_weights/qwen35-4b-combined-final',
}

for name, tinker_path in SFT_CHECKPOINTS.items():
    lora_dir = f'{DRIVE}/sft_lora_{name}'
    if os.path.exists(lora_dir) and any(f.endswith('.safetensors') for f in os.listdir(lora_dir)):
        print(f'{name}: already downloaded at {lora_dir}')
        continue
    print(f'Downloading {name}...')
    url_resp = rest.get_checkpoint_archive_url_from_tinker_path(tinker_path).result()
    local_tar = f'/tmp/lora_{name}.tar'
    r = requests.get(url_resp.url, stream=True)
    with open(local_tar, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    os.makedirs(lora_dir, exist_ok=True)
    with tarfile.open(local_tar) as t:
        t.extractall(lora_dir)
    print(f'  Saved: {lora_dir} ({os.listdir(lora_dir)})')

print('All SFT LoRA weights downloaded!')

In [ ]:
# Cell 2: GRPO on GSM8K for each SFT model
# Following HuggingFace/Unsloth GRPO guide:
# https://huggingface.co/learn/llm-course/en/chapter12/6

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset, Dataset
import torch, re

max_seq_length = 2048  # Our SFT training data had median 824 tokens, max ~5000
lora_rank = 32

SYSTEM_PROMPT = """Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>"""

# ── Dataset ──
def extract_hash_answer(text):
    if '####' not in text: return None
    return text.split('####')[1].strip()

def get_gsm8k_questions(split='train'):
    data = load_dataset('openai/gsm8k', 'main')[split]
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']},
        ],
        'answer': extract_hash_answer(x['answer']),
    })
    return data

dataset = get_gsm8k_questions()
print(f'GRPO dataset: {len(dataset)} GSM8K train problems')

# ── Reward Functions ──
def extract_xml_answer(text):
    if '<answer>' in text and '</answer>' in text:
        return text.split('<answer>')[-1].split('</answer>')[0].strip()
    # Fallback: last number
    numbers = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return numbers[-1] if numbers else ''

def correctness_reward(prompts, completions, answer, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    return [2.0 if e == a else 0.0 for e, a in zip(extracted, answer)]

def format_reward(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    return [0.5 if '<reasoning>' in r and '</reasoning>' in r and '<answer>' in r else 0.0 for r in responses]

def int_reward(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.replace('-','').replace('.','').isdigit() else 0.0 for r in extracted]

print('Reward functions defined')

In [ ]:
# Cell 3: Run GRPO for each SFT model
# Saves GRPO'd LoRA weights to Drive after each

for teacher in ['kimi', 'glm5', 'combined']:
    grpo_dir = f'{DRIVE}/grpo_lora_{teacher}'
    if os.path.exists(grpo_dir) and os.listdir(grpo_dir):
        print(f'{teacher}: GRPO already done, skipping')
        continue
    
    print(f'\n{"="*60}')
    print(f'GRPO: {teacher}')
    print(f'{"="*60}')
    
    sft_dir = f'{DRIVE}/sft_lora_{teacher}'
    
    # Load base model + SFT LoRA
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=sft_dir,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        fast_inference=True,
        max_lora_rank=lora_rank,
        gpu_memory_utilization=0.6,
    )
    
    # Configure GRPO
    training_args = GRPOConfig(
        learning_rate=5e-6,
        adam_beta1=0.9,
        adam_beta2=0.99,
        weight_decay=0.1,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        optim='paged_adamw_8bit',
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=8,
        max_prompt_length=512,                           # GSM8K questions + system prompt
        max_completion_length=max_seq_length - 512,      # 1536 tokens for reasoning + answer
        max_steps=250,
        save_steps=250,
        max_grad_norm=0.1,
        report_to='none',
        output_dir=f'/tmp/grpo_{teacher}',
    )
    
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[correctness_reward, format_reward, int_reward],
        args=training_args,
        train_dataset=dataset,
    )
    
    print(f'  Training (250 steps)...')
    trainer.train()
    
    # Save GRPO'd LoRA to Drive
    model.save_lora(grpo_dir)
    print(f'  Saved GRPO LoRA: {grpo_dir}')
    
    del model, tokenizer, trainer
    torch.cuda.empty_cache()

print('\nAll GRPO training complete!')

In [ ]:
# Cell 4: Merge all models for lm-eval
# lm-eval needs full merged models, not LoRA adapters

from unsloth import FastLanguageModel
import torch

MODELS_TO_MERGE = {}

# SFT models
for teacher in ['glm5', 'kimi', 'combined']:
    sft_dir = f'{DRIVE}/sft_lora_{teacher}'
    merged_dir = f'{DRIVE}/merged_sft_{teacher}'
    if os.path.exists(sft_dir):
        MODELS_TO_MERGE[f'sft-{teacher}'] = (sft_dir, merged_dir)

# GRPO models
for teacher in ['glm5', 'kimi', 'combined']:
    grpo_dir = f'{DRIVE}/grpo_lora_{teacher}'
    merged_dir = f'{DRIVE}/merged_grpo_{teacher}'
    if os.path.exists(grpo_dir):
        MODELS_TO_MERGE[f'grpo-{teacher}'] = (grpo_dir, merged_dir)

for name, (lora_dir, merged_dir) in MODELS_TO_MERGE.items():
    if os.path.exists(merged_dir) and os.listdir(merged_dir):
        print(f'{name}: already merged')
        continue
    print(f'Merging {name}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_dir, max_seq_length=4096, dtype=None, load_in_4bit=False,
    )
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    del model, tokenizer
    torch.cuda.empty_cache()
    print(f'  Saved: {merged_dir}')

print('All models merged!')

In [ ]:
# Cell 5: Run lm-evaluation-harness on ALL models
# Industry standard — same tool used by HuggingFace Open LLM Leaderboard
import subprocess

ALL_TASKS = 'gsm8k_cot,minerva_math,arc_challenge,gpqa_diamond,mmlu_pro'

# Build model list
EVAL_MODELS = [
    # (name, path, is_chat)
    ('base-4b', 'Qwen/Qwen3.5-4B', True),
    ('llama-3b', 'meta-llama/Llama-3.2-3B', False),
    ('qwen3-8b', 'Qwen/Qwen3-8B', False),
    ('gpt-oss-20b', 'openai/gpt-oss-20b', True),  # Fits on H100 80GB
]

# Add merged SFT + GRPO models
for name, (_, merged_dir) in MODELS_TO_MERGE.items():
    if os.path.exists(merged_dir):
        EVAL_MODELS.append((name, merged_dir, True))

print(f'{len(EVAL_MODELS)} models to evaluate:')
for name, path, chat in EVAL_MODELS:
    print(f'  {name}')

for name, model_path, is_chat in EVAL_MODELS:
    result_dir = f'{DRIVE}/eval_results/{name}'
    if os.path.exists(f'{result_dir}/results.json'):
        print(f'{name}: already evaluated')
        continue
    
    print(f'\nEvaluating: {name}...')
    cmd = [
        'lm-eval', '--model', 'hf',
        '--model_args', f'pretrained={model_path},dtype=bfloat16,trust_remote_code=True',
        '--tasks', ALL_TASKS,
        '--batch_size', 'auto',
        '--output_path', result_dir,
        '--log_samples',
    ]
    if is_chat:
        cmd.extend(['--apply_chat_template', '--fewshot_as_multiturn'])
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✅ Done')
    else:
        print(f'  ❌ Error: {result.stderr[-500:]}')
    # Print last few lines of output for monitoring
    print(result.stdout[-500:])

In [ ]:
# Cell 6: Collect and display all results
import glob, json

all_results = {}
for name, _, _ in EVAL_MODELS:
    result_files = glob.glob(f'{DRIVE}/eval_results/{name}/**/results.json', recursive=True)
    if not result_files:
        continue
    with open(result_files[0]) as f:
        data = json.load(f)
    results = {}
    for task_name, task_data in data.get('results', {}).items():
        for metric in ['acc,none', 'acc_norm,none', 'exact_match,strict-match',
                       'exact_match,flexible-extract', 'acc,none']:
            if metric in task_data:
                results[task_name] = round(task_data[metric] * 100, 1)
                break
    all_results[name] = results

# Print table
benchmarks = ['gsm8k_cot', 'minerva_math', 'arc_challenge', 'gpqa_diamond', 'mmlu_pro']
labels = ['GSM8K', 'MATH', 'ARC', 'GPQA', 'MMLU-Pro']

print(f'{"Model":25s}', end='')
for label in labels:
    print(f'{label:>10s}', end='')
print()
print('-' * (25 + 10 * len(labels)))
for name, _, _ in EVAL_MODELS:
    if name not in all_results:
        continue
    print(f'{name:25s}', end='')
    for bench in benchmarks:
        score = all_results[name].get(bench, '—')
        s = f'{score}%' if isinstance(score, (int, float)) else str(score)
        print(f'{s:>10s}', end='')
    print()

with open(f'{DRIVE}/eval_results/comparison.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved: {DRIVE}/eval_results/comparison.json')

In [ ]:
# Cell 7: Trick questions — side by side (manual generation, not lm-eval)
from unsloth import FastLanguageModel
import torch

TRICKS = [
    ('A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?', '9'),
    ('If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?', '5 minutes'),
    ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?', '$0.05'),
]

SYS = 'You are a helpful reasoning assistant. Think through problems step by step.'

# Test on a few key models
for model_name in ['base-4b', 'sft-kimi', 'grpo-kimi']:
    path = next((p for n, p, _ in EVAL_MODELS if n == model_name), None)
    if not path: continue
    model, tok = FastLanguageModel.from_pretrained(model_name=path, max_seq_length=2048, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    print(f'\n=== {model_name} ===')
    for q, expected in TRICKS:
        msgs = [{'role':'system','content':SYS}, {'role':'user','content':q}]
        inp = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
        out = model.generate(inp, max_new_tokens=512, temperature=0.6, do_sample=True)
        resp = tok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)
        print(f'  Q: {q[:60]}...')
        print(f'  Expected: {expected}')
        print(f'  Model: {resp[:200]}')
        print()
    del model, tok; torch.cuda.empty_cache()

In [ ]:
# Cell 8: Push best model to HuggingFace
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

# Pick best based on GSM8K
distilled = {k:v for k,v in all_results.items() if 'sft' in k or 'grpo' in k}
best = max(distilled, key=lambda k: distilled[k].get('gsm8k_cot', 0))
print(f'Best model: {best}')
print(f'Scores: {distilled[best]}')

best_dir = f'{DRIVE}/merged_{best}'
model, tok = FastLanguageModel.from_pretrained(model_name=best_dir, max_seq_length=4096, load_in_4bit=False)

model.push_to_hub_merged(
    'bmeyer2025/qwen3.5-4b-reasoning-distilled', tok,
    save_method='merged_16bit', token=userdata.get('HF_TOKEN'),
)
print('Pushed to HuggingFace!')

In [ ]:
# Cell 9: GGUF export
model.push_to_hub_gguf(
    'bmeyer2025/qwen3.5-4b-reasoning-distilled', tok,
    quantization_method=['q4_k_m', 'q8_0'],
    token=userdata.get('HF_TOKEN'),
)
print('GGUF pushed!')
print('Run: ollama run hf.co/bmeyer2025/qwen3.5-4b-reasoning-distilled:Q4_K_M')